### Scraping the Craigslist

In [77]:
# Creating Top 100 most frequent locations with ZIP CODES, from the 'ucla_final.csv' file.
import pandas as pd

# Read the CSV file
df = pd.read_csv("ucla_final.csv", low_memory=False)  # low_memory=False because it was trying to guess the data types.

# Turning zip codes into integer
df['postal_code'] = df['postal_code'].apply( lambda x: str(int(x)) if pd.notnull(x) else None )

# Count the top 150 most frequent names
top_150_locations = df['name'].value_counts().head(150)

# Extract ZIP code information (dropping duplicates)
zip_map = df[['name', 'postal_code']].drop_duplicates().set_index('name')

# Join location counts with ZIP codes
result_df = top_150_locations.to_frame(name='Count').join(zip_map, how='left')

# Clean the columns
result_df.reset_index(inplace=True)
result_df.rename(columns={'index': 'Location', 'postal_code': 'Zip_Code'}, inplace=True)

# Display the result
locations_to_scrape_df = result_df.copy()
display(locations_to_scrape_df)


,name,Count,Zip_Code
0,agoura hills,162,91301
1,pico rivera,162,None
2,azusa,162,None
3,hawaiian gardens,162,None
4,hawthorne,162,None
...,...,...,...
145,los feliz,81,90027
146,lopez/kagel canyons,81,91342
147,littlerock,81,93543
148,lincoln heights,81,None


In [79]:
# Function that scrapes the craigslist based on the name and postal code,
# if no postal code present, it only queries the name of the location.
import requests
from bs4 import BeautifulSoup
import time
import math

def scrape_craigslist(name, postal_code=None, sleep_time=1):
    base_url = "https://losangeles.craigslist.org/search/hhh"
    listings_per_page = 120
    seen_links = set()
    all_listings = []
    count = 0

    # Initial parameters
    params = {"query": name, "s": 0}  # "s" is for start, s = 0 means starting with the first 120 listings.
    if postal_code:
        params["postal"] = str(postal_code)

    # Get the first page
    res = requests.get(base_url, params=params)
    soup = BeautifulSoup(res.text, "html.parser")

    # Parse total count and extract total listing count from `visible-counts`
    visible_counts = soup.find("div", class_="visible-counts")
    total_text = visible_counts.get_text(strip=True) if visible_counts else ""
    try:
        total_listings = int(total_text.split("of")[-1].replace(",", "").strip())
    except:
        total_listings = 1200  # default max

    max_pages = int(total_listings / listings_per_page)

    # Loop through pages and scrape listings
    for page in range(max_pages):
        params["s"] = page * listings_per_page
        res = requests.get(base_url, params=params)
        soup = BeautifulSoup(res.text, "html.parser")
        listings = soup.find_all("li", class_="cl-static-search-result")

        if not listings:
            break

        for listing in listings:
            a_tag = listing.find("a", href=True)
            title_div = a_tag.find("div", class_="title") if a_tag else None
            if title_div and a_tag:
                link = a_tag["href"]
                if link in seen_links:
                    continue
                title = title_div.get_text(strip=True)
                all_listings.append({
                    "title": title,
                    "link": link,
                    "search_name": name,
                    "search_zip": postal_code
                })
                seen_links.add(link)
                count += 1
                print(f"[{count}] {title} — {link}")

        time.sleep(sleep_time)

    print(f"\n Done: {count} listings scraped for {name} ({postal_code})")
    return all_listings


In [59]:
# Example on how the function works
#results = scrape_craigslist(name="West Adams", postal_code="90007")

results = scrape_craigslist(name="pico rivera")


[1] 🏠🏠SE RENTA CUARTO PARA MUJER SOLA O CON NINO 🏠🏠 — https://losangeles.craigslist.org/sgv/roo/d/pico-rivera-se-renta-cuarto-para-mujer/7845563021.html
[2] Room with private bathroom — https://losangeles.craigslist.org/sgv/roo/d/pico-rivera-room-with-private-bathroom/7845454406.html
[3] Large 2bed 2bath Unit-Comfort And Affordability As You Deserve! Call! — https://losangeles.craigslist.org/lac/apa/d/pico-rivera-large-2bed-2bath-unit/7845448251.html
[4] Charming 2 Bed/2 Bath Single Family Home in Pico Rivera - $5500/mo — https://losangeles.craigslist.org/wst/apa/d/pico-rivera-charming-bed-bath-single/7845383266.html
[5] Shared Room for rent Not Private — https://losangeles.craigslist.org/lac/roo/d/pico-rivera-shared-room-for-rent-not/7845357022.html
[6] What a great place to live! 2 bed, 2 bath, 1000sq ft. Come home today! — https://losangeles.craigslist.org/lac/apa/d/pico-rivera-what-great-place-to-live/7845315167.html
[7] This 2bed/2bath Won't Last Long-Double Closet-Double Comfort-

In [81]:
# Using the function with result_df dataframe to get the names and urls of houses for the first 150 most frequent places

all_results = []

for _, row in result_df.iterrows():
    name = row['name']
    postal_code = row['Zip_Code']

    try:
        if pd.notnull(postal_code): #if postal code exists
            listings = scrape_craigslist(name=name, postal_code=postal_code)
        else:
            listings = scrape_craigslist(name=name)
        all_results.extend(listings)
    except Exception as e:
        print(f" Error scraping {name} ({postal_code}): {e}")

# Optional: Convert to DataFrame
listings_df = pd.DataFrame(all_results)


[1] Bask in luxury at The Lexington Apartment Homes. The finest in upscale — https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-bask-in-luxury-at-the/7844882251.html
[2] Dreams find their home here: Book our sought-after 1 BR, 1 BA now! — https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-dreams-find-their-home/7839104974.html
[3] Your haven from a busy lifestyle! Calm and quiet 1 bed / 1 bath — https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-your-haven-from-busy/7844882289.html
[4] Your haven from a busy lifestyle! Calm and quiet 1 bed / 1 bath — https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-your-haven-from-busy/7843369268.html
[5] Beautiful 1 bed / 1 bath with gorgeous grounds and landscaping — https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-beautiful-bed-bath-with/7843369358.html
[6] Serene garden settings with beautiful views! 763 SqFt 1 BR / 1 BA! — https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-serene-garden-settings/7843369508.h

In [15]:
# Data about scraping the urls and names of lists
########FIRST 50 ROWS###########
# started at 4:24
# finished at 4:36
################################ first 50 rows took 12 minutes and generated 9353 houses.

########NEXT 50 ROWS###########
# started at 5:58
# finished at 6:12 OR SOMEWHERE B/W 6:10 AND 6:12
################################ assuming it will at maximum take 14 minutes.
### Estimated Total amount of run time: 27-30 minutes.

In [83]:
print(len(listings_df))
display(listings_df)

# Save 'listings_df' into a csv file.
listings_df.to_csv('craigslist_listings.csv')


23283


,title,link,search_name,search_zip
0,Bask in luxury at The Lexington Apartment Home...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301
1,Dreams find their home here: Book our sought-a...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301
2,Your haven from a busy lifestyle! Calm and qui...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301
3,Your haven from a busy lifestyle! Calm and qui...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301
4,Beautiful 1 bed / 1 bath with gorgeous grounds...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301
...,...,...,...,...
23278,COMMERCIAL WAREHOUSE FOR LEASE,https://losangeles.craigslist.org/sgv/off/d/el...,leona valley,None
23279,NEED WAREHOUSE SPACE?,https://losangeles.craigslist.org/lac/off/d/lo...,leona valley,None
23280,COMMERCIAL WAREHOUSE FOR LEASE,https://losangeles.craigslist.org/lac/off/d/sa...,leona valley,None
23281,COMMERCIAL WAREHOUSE FOR LEASE,https://losangeles.craigslist.org/lac/off/d/co...,leona valley,None


## Now it is time to scrape the content of the houses from the URLs we have scraped

In [3]:
# Creating the function to scrape the URLs and create the features of the houses

import requests
from bs4 import BeautifulSoup
import re
import sys

def scrape_craigslist_listing(url):

    try:
        r = requests.get(url)
        r.raise_for_status()
    except:
        sys.stderr.write(f'Error: The url {url} could not be requested properly!\n')
        return None
        
    soup = BeautifulSoup(r.text, 'html.parser')

    # Title
    title_tag = soup.find('span', id='titletextonly')
    if title_tag:
        title_main = title_tag.get_text(strip=True)
        try:
            location = title_tag.find_next_sibling('span')
            location_text = location.get_text(strip=True) if location else ''
        except:
            location_text = ''
        #location = title_tag.find_next_sibling('span')
        #location_text = location.get_text(strip=True) if location else ''
        title = f"{title_main} {location_text}".strip()
    else:
        title = None

    # Price
    price_tag = soup.find('span', class_='price')
    price = price_tag.get_text(strip=True) if price_tag else None

    # Address
    address_tag = soup.find('h2', class_='street-address')
    address = address_tag.get_text(strip=True) if address_tag else None

    # Description
    desc_tag = soup.find('section', id='postingbody')
    description = desc_tag.get_text(separator='\n', strip=True) if desc_tag else None

    # Coordinates
    map_tag = soup.find('div', id='map')
    latitude = map_tag.get('data-latitude') if map_tag else None
    longitude = map_tag.get('data-longitude') if map_tag else None

    # Bedrooms, Bathrooms, and SqFt from class="attr important"
    bedrooms = bathrooms = sqft = None
    important_attrs = soup.select('span.attr.important')
    for attr in important_attrs:
        text = attr.get_text(strip=True)
        
        # Match bedroom and bathroom like "1BR / 1Ba"
        if 'br' in text.lower() and 'ba' in text.lower(): 
            # Remember it needs to be lower case!
            br_ba_match = re.search(r'(\d+)\s*br\s*/\s*(\d+)\s*ba', text.lower() )            
            if br_ba_match:
                bedrooms = br_ba_match.group(1)
                bathrooms = br_ba_match.group(2)
        
        # Match square footage like "763ft"
        elif 'ft' in text.lower():
            sqft_match = re.search(r'(\d+)\s*ft', text, re.IGNORECASE)
            if sqft_match:
                sqft = sqft_match.group(1)

    return {
        'title': title,
        'price': price,
        'bedrooms': bedrooms,
        'bathrooms': bathrooms,
        'square_feet': sqft,
        'address': address,
        'latitude': latitude,
        'longitude': longitude,
        'description': description,
    }


In [129]:
df_links = pd.read_csv("craigslist_listings.csv") 

display(df_links)

,Unnamed: 0,title,link,search_name,search_zip
0,0,Bask in luxury at The Lexington Apartment Home...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301.0
1,1,Dreams find their home here: Book our sought-a...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301.0
2,2,Your haven from a busy lifestyle! Calm and qui...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301.0
3,3,Your haven from a busy lifestyle! Calm and qui...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301.0
4,4,Beautiful 1 bed / 1 bath with gorgeous grounds...,https://losangeles.craigslist.org/sfv/apa/d/ag...,agoura hills,91301.0
...,...,...,...,...,...
23278,23278,COMMERCIAL WAREHOUSE FOR LEASE,https://losangeles.craigslist.org/sgv/off/d/el...,leona valley,NaN
23279,23279,NEED WAREHOUSE SPACE?,https://losangeles.craigslist.org/lac/off/d/lo...,leona valley,NaN
23280,23280,COMMERCIAL WAREHOUSE FOR LEASE,https://losangeles.craigslist.org/lac/off/d/sa...,leona valley,NaN
23281,23281,COMMERCIAL WAREHOUSE FOR LEASE,https://losangeles.craigslist.org/lac/off/d/co...,leona valley,NaN


In [5]:
# TESTING OUT THE FUNCTION, SCRAPING THE FIRST 10 ROWS AND DISPLAYING IT:
###
# Using the function - Be careful to arrange which house range to scrape before runnning the code!
###
import time
import pandas as pd 

# Loading the csv with links
df_links = pd.read_csv("craigslist_listings.csv") 

# Scrape listings
all_results = []
save_every = 200
for i, row in df_links.iloc[0:10].iterrows():
    url = row['link']
    print(f"Scraping {i+1}/{len(df_links)}: {url}")
    result = scrape_craigslist_listing(url)
    if result:
        result['search_name'] = row.get('search_name')
        result['search_zip'] = row.get('search_zip')
        result['url'] = url
        all_results.append(result)
    time.sleep(1.2)  # TO be politely scraping

# Convert to DataFrame and save
df_results_test = pd.DataFrame(all_results)
display(df_results_test)

Scraping 1/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-bask-in-luxury-at-the/7844882251.html
Scraping 2/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-dreams-find-their-home/7839104974.html


Error: The url https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-dreams-find-their-home/7839104974.html could not be requested properly!


Scraping 3/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-your-haven-from-busy/7844882289.html
Scraping 4/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-your-haven-from-busy/7843369268.html
Scraping 5/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-beautiful-bed-bath-with/7843369358.html
Scraping 6/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-serene-garden-settings/7843369508.html
Scraping 7/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-explore-dream-settle-your/7844882343.html
Scraping 8/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-763-sqft-bd-ba-ready-to/7843369548.html
Scraping 9/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-bask-in-luxury-at-the/7842857262.html
Scraping 10/23283: https://losangeles.craigslist.org/sfv/apa/d/agoura-hills-beautiful-bed-bath-with/7842857122.html


,title,price,bedrooms,bathrooms,square_feet,address,latitude,longitude,description,search_name,search_zip,url
0,Bask in luxury at The Lexington Apartment Home...,"$2,561",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
1,Your haven from a busy lifestyle! Calm and qui...,"$2,561",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
2,Your haven from a busy lifestyle! Calm and qui...,"$2,814",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
3,Beautiful 1 bed / 1 bath with gorgeous grounds...,"$2,814",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
4,Serene garden settings with beautiful views! 7...,"$2,814",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
5,"Explore, dream, settle. Your new journey start...","$2,561",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
6,"763 sqft 1 bd, 1 ba ready to lease! Come on ov...","$2,814",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
7,Bask in luxury at The Lexington Apartment Home...,"$2,814",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
8,Beautiful 1 bed / 1 bath with gorgeous grounds...,"$2,814",1,1,763,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...


In [131]:
###
# Using the function - Be careful to arrange which house range to scrape before runnning the code!
###
import time
import pandas as pd 

# Loading the csv with links
df_links = pd.read_csv("craigslist_listings.csv") 

# Scrape listings
all_results = []
save_every = 200
for i, row in df_links.iloc[20000:23280].iterrows(): # DON'T FORGET TO ARRANGE THE NUMBER OF ROWS!!!!!
    url = row['link']
    print(f"Scraping {i+1}/{len(df_links)}: {url}")
    result = scrape_craigslist_listing(url)
    if result:
        result['search_name'] = row.get('search_name')
        result['search_zip'] = row.get('search_zip')
        result['url'] = url
        all_results.append(result)

    '''
    # Save progress every save_every (N) listings. This is to save us when website restricts the access
    if (i + 1) % save_every == 0:
        temp_df = pd.DataFrame(all_results)
        temp_df.to_csv("scraped_progress.csv", index=False)
        print(f" Saved progress at {i+1} listings")
    '''
    time.sleep(1.2)  # TO be politely scraping

# Convert to DataFrame and save
df_results = pd.DataFrame(all_results)
display(df_results)
#df_results.to_csv("scraped_craigslist_details.csv", index=False)


Scraping 20001/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-kayak-and-individual/7844890638.html
Scraping 20002/23283: https://losangeles.craigslist.org/wst/vac/d/venice-beachfront-penthouse-marina-del/7840870931.html
Scraping 20003/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-beautiful-720-sq-ft-bed/7843737119.html
Scraping 20004/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-state-of-the-art-fitness/7844873987.html


Error: The url https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-state-of-the-art-fitness/7844873987.html could not be requested properly!


Scraping 20005/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-style-and-luxury-at/7842727350.html
Scraping 20006/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-spacious-great-natural/7844167596.html
Scraping 20007/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-zoom-on-over-for-look-at/7842146862.html
Scraping 20008/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-we-welcome-your-furry/7840559435.html
Scraping 20009/23283: https://losangeles.craigslist.org/wst/reo/d/marina-del-rey-land-for-sale-roughly-an/7844789964.html
Scraping 20010/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-ample-closets-swimming/7841209298.html
Scraping 20011/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-up-to-7k-off-rent-for/7843740096.html
Scraping 20012/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-parcel-lockers/7844745557.html
Scraping 20013/23283: https://losangeles.craigsli

Error: The url https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-zoom-on-over-for-look-at/7837923310.html could not be requested properly!


Scraping 20214/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-we-welcome-your-furry/7841487555.html
Scraping 20215/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-ample-closets-nature/7841462693.html
Scraping 20216/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-up-to-7k-off-rent-for/7840920719.html
Scraping 20217/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-up-to-7k-off-rent-for/7840899211.html
Scraping 20218/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-short-term-leases/7840884014.html
Scraping 20219/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-short-term-leases/7840861877.html
Scraping 20220/23283: https://losangeles.craigslist.org/wst/apa/d/marina-del-rey-welcome-to-your-new-bed/7841381560.html
Scraping 20221/23283: https://losangeles.craigslist.org/sfv/apa/d/north-hollywood-spacious-2bed1bath/7845591622.html
Scraping 20222/23283: https://losangeles.craigslist.org/s

...

Error: The url https://losangeles.craigslist.org/lac/apa/d/los-angeles-classic-1950s-courtyard/7841218813.html could not be requested properly!


Scraping 23070/23283: https://losangeles.craigslist.org/lac/roo/d/los-angeles-private-bedroom-in-the/7838778658.html
Scraping 23071/23283: https://losangeles.craigslist.org/lac/sub/d/los-angeles-br-in-the-heart-of-los/7840834140.html
Scraping 23072/23283: https://losangeles.craigslist.org/lac/sub/d/los-angeles-master-bedroom-for-the-rest/7840709519.html
Scraping 23073/23283: https://losangeles.craigslist.org/lac/roo/d/los-angeles-short-term-master-bedroom/7840684132.html
Scraping 23074/23283: https://losangeles.craigslist.org/lac/apa/d/los-angeles-all-new-charming-bright-11/7840652310.html
Scraping 23075/23283: https://losangeles.craigslist.org/lac/apa/d/los-angeles-breathtaking-remodeled-1br/7838552952.html
Scraping 23076/23283: https://losangeles.craigslist.org/lac/apa/d/spanish-house-for-filming-shoot/7844893433.html
Scraping 23077/23283: https://losangeles.craigslist.org/lac/off/d/los-angeles-prime-retail-space-in-high/7839369500.html
Scraping 23078/23283: https://losangeles.craigs

Error: The url https://losangeles.craigslist.org/lac/roo/d/los-angeles-subletting-private-bedroom/7837982824.html could not be requested properly!


Scraping 23083/23283: https://losangeles.craigslist.org/ant/sub/d/lancaster-se-renta-traila/7840719250.html
Scraping 23084/23283: https://losangeles.craigslist.org/ant/sub/d/lancaster-se-renta-traila/7844653053.html
Scraping 23085/23283: https://losangeles.craigslist.org/ant/roo/d/littlerock-room-for-rent/7844788846.html
Scraping 23086/23283: https://losangeles.craigslist.org/ant/reo/d/littlerock-25-acres-lot-on-144-east-ave/7844644820.html
Scraping 23087/23283: https://losangeles.craigslist.org/ant/roo/d/littlerock-room-rent/7844637825.html
Scraping 23088/23283: https://losangeles.craigslist.org/sfv/off/d/littlerock-sba-restaurant/7843207416.html
Scraping 23089/23283: https://losangeles.craigslist.org/ant/off/d/littlerock-sqft-single-commercial-bldg/7840328309.html
Scraping 23090/23283: https://losangeles.craigslist.org/ant/reb/d/littlerock-land-for-sale/7838972373.html
Scraping 23091/23283: https://losangeles.craigslist.org/sfv/reo/d/palmdale-8966-ac-cherry-orchard-horse/7841981176.h

,title,price,bedrooms,bathrooms,square_feet,address,latitude,longitude,description,search_name,search_zip,url
0,"Kayak and Individual Storage Available, Busine...","$5,751",3,2,None,"13488 W Maxella Ave, Marina Del Rey, CA 90292",33.986223,-118.442466,QR Code Link to This Post\nStella Apartments\n...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/apa/d/ma...
1,Beachfront Penthouse Marina Del Rey (Venice),"$3,200",1,1,1100,"4121 via marina, marina del rey, 90292., Venic...",33.980200,-118.460400,QR Code Link to This Post\nThis is a really aw...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/vac/d/ve...
2,Beautiful 720 Sq Ft 1 Bed / 1 Bath close to ca...,"$3,365",1,1,720,"4108 Del Rey Ave, Marina del Rey, CA 90292",33.988362,-118.444979,QR Code Link to This Post\nCome home to a grea...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/apa/d/ma...
3,Style and Luxury at Tribeca Urban Apartments. ...,"$3,365",1,1,720,"4108 Del Rey Ave, Marina del Rey, CA 90292",33.988362,-118.444979,QR Code Link to This Post\n-- Tribeca Urban Ap...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/apa/d/ma...
4,"Spacious, great natural light, large balcony -...","$3,430",1,1,831,"4600 Vía Marina, Marina del Rey, CA 90292",33.977900,-118.452500,QR Code Link to This Post\nGET TWO MONTHS FREE...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/apa/d/ma...
...,...,...,...,...,...,...,...,...,...,...,...,...
3183,Build Business Credibility with a Virtual Offi...,None,None,None,None,"3415 Sepulveda, Culver City, CA 90034",34.012711,-118.417311,QR Code Link to This Post\nContact Cat Soliman...,lincoln heights,NaN,https://losangeles.craigslist.org/wst/off/d/lo...
3184,LARGE LAND NEEDED Palmdale Lancaster Littleroc...,None,None,None,None,None,34.710859,-118.326702,QR Code Link to This Post\n________________LAR...,leona valley,NaN,https://losangeles.craigslist.org/ant/reb/d/la...
3185,COMMERCIAL WAREHOUSE FOR LEASE (Los Angeles Co...,None,None,None,None,None,34.096400,-117.968200,QR Code Link to This Post\nDon’t have time to ...,leona valley,NaN,https://losangeles.craigslist.org/sgv/off/d/ba...
3186,COMMERCIAL WAREHOUSE FOR LEASE (Los Angeles Co...,None,None,None,None,None,34.079100,-118.037100,QR Code Link to This Post\nDon’t have time to ...,leona valley,NaN,https://losangeles.craigslist.org/sgv/off/d/el...


In [133]:
import pandas as pd

df_results_1 = pd.DataFrame(all_results)
df_results = df_results_1.copy()

In [135]:
# Saving the Results
import pandas as pd

#display(df_results)

# Price needs processing
def clean_price(p):
    try:
        return (p.replace("$", ""))
    except:
        return None
df_results['price'] = df_results['price'].apply(clean_price)

display(df_results)

df_results.to_csv("houses_new_20000_23280.csv", index=False)


,title,price,bedrooms,bathrooms,square_feet,address,latitude,longitude,description,search_name,search_zip,url
0,"Kayak and Individual Storage Available, Busine...","5,751",3,2,None,"13488 W Maxella Ave, Marina Del Rey, CA 90292",33.986223,-118.442466,QR Code Link to This Post\nStella Apartments\n...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/apa/d/ma...
1,Beachfront Penthouse Marina Del Rey (Venice),"3,200",1,1,1100,"4121 via marina, marina del rey, 90292., Venic...",33.980200,-118.460400,QR Code Link to This Post\nThis is a really aw...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/vac/d/ve...
2,Beautiful 720 Sq Ft 1 Bed / 1 Bath close to ca...,"3,365",1,1,720,"4108 Del Rey Ave, Marina del Rey, CA 90292",33.988362,-118.444979,QR Code Link to This Post\nCome home to a grea...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/apa/d/ma...
3,Style and Luxury at Tribeca Urban Apartments. ...,"3,365",1,1,720,"4108 Del Rey Ave, Marina del Rey, CA 90292",33.988362,-118.444979,QR Code Link to This Post\n-- Tribeca Urban Ap...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/apa/d/ma...
4,"Spacious, great natural light, large balcony -...","3,430",1,1,831,"4600 Vía Marina, Marina del Rey, CA 90292",33.977900,-118.452500,QR Code Link to This Post\nGET TWO MONTHS FREE...,marina del rey,90292.0,https://losangeles.craigslist.org/wst/apa/d/ma...
...,...,...,...,...,...,...,...,...,...,...,...,...
3183,Build Business Credibility with a Virtual Offi...,None,None,None,None,"3415 Sepulveda, Culver City, CA 90034",34.012711,-118.417311,QR Code Link to This Post\nContact Cat Soliman...,lincoln heights,NaN,https://losangeles.craigslist.org/wst/off/d/lo...
3184,LARGE LAND NEEDED Palmdale Lancaster Littleroc...,None,None,None,None,None,34.710859,-118.326702,QR Code Link to This Post\n________________LAR...,leona valley,NaN,https://losangeles.craigslist.org/ant/reb/d/la...
3185,COMMERCIAL WAREHOUSE FOR LEASE (Los Angeles Co...,None,None,None,None,None,34.096400,-117.968200,QR Code Link to This Post\nDon’t have time to ...,leona valley,NaN,https://losangeles.craigslist.org/sgv/off/d/ba...
3186,COMMERCIAL WAREHOUSE FOR LEASE (Los Angeles Co...,None,None,None,None,None,34.079100,-118.037100,QR Code Link to This Post\nDon’t have time to ...,leona valley,NaN,https://losangeles.craigslist.org/sgv/off/d/el...


In [137]:
# merging the first 2 ten thousand houses.
import pandas as pd

# Load the two datasets
df1 = pd.read_csv("houses_new_0_10000.csv")
df2 = pd.read_csv("houses_new_10000_20000.csv")
df3 = pd.read_csv("houses_new_20000_23280.csv")

# Merge row-wise
merged_df = pd.concat([df1, df2, df3], ignore_index=True)

# Save to the new CSV
merged_df.to_csv("houses_new_0_23280.csv", index=False)


In [157]:
import pandas as pd

houses_df = pd.read_csv("houses_new_0_23280.csv", low_memory=False)

# Step 1: Remove exact duplicate listings (by address + price only)
houses_df = houses_df.drop_duplicates(subset=['address', 'price'], keep='first')

# Step 2: Drop houses missing too much information
important_fields = ['square_feet', 'bedrooms', 'bathrooms', 'address', 'search_name', 'search_zip']
threshold = 2  # Allow up to 4 missing important fields

houses_df['missing_count'] = houses_df[important_fields].isna().sum(axis=1)
houses_df_cleaned = houses_df[houses_df['missing_count'] <= threshold].drop(columns=['missing_count'])

# Step 3: Minor Preprocessing
#houses_df_cleaned['square_feet'] = pd.to_numeric(houses_df_cleaned['square_feet'], errors='coerce')
#houses_df_cleaned['bedrooms'] = pd.to_numeric(houses_df_cleaned['bedrooms'], errors='coerce')
#houses_df_cleaned['bathrooms'] = pd.to_numeric(houses_df_cleaned['bathrooms'], errors='coerce')
#houses_df_cleaned['search_zip'] = pd.to_numeric(houses_df_cleaned['search_zip'], errors='coerce')

houses_df_cleaned = houses_df_cleaned.reset_index(drop=True)

print(f"Original houses: {len(houses_df)}")
print(f"After cleaning: {len(houses_df_cleaned)}")
display(houses_df_cleaned.head())

houses_df_cleaned.to_csv("houses_cleaned.csv", index=False)

# Saving the final file into Predicting_Energy
houses_df_cleaned.to_csv('../Predicting_Energy/houses_cleaned.csv', index=False)


Original houses: 4928
After cleaning: 3597


,title,price,bedrooms,bathrooms,square_feet,address,latitude,longitude,description,search_name,search_zip,url
0,Bask in luxury at The Lexington Apartment Home...,"2,561",1.0,1.0,763.0,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
1,Dreams find their home here: Book our sought-a...,"2,808",1.0,1.0,763.0,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
2,Your haven from a busy lifestyle! Calm and qui...,"2,814",1.0,1.0,763.0,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
3,Your haven from a busy lifestyle! Calm and qui...,"2,770",1.0,1.0,763.0,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
4,Your haven from a busy lifestyle! Calm and qui...,"2,723",1.0,1.0,763.0,"30856 Agoura Rd, Agoura Hills, CA 91301",34.143276,-118.794957,QR Code Link to This Post\nThe Lexington Apart...,agoura hills,91301.0,https://losangeles.craigslist.org/sfv/apa/d/ag...
